# 02 - Climate predictors (temperature, precipitation, freezing level)

**Reviewer point addressed:** *"correlate snow trends with temperature, precipitation ..."*

Two ways to get the predictors; run **either** section 1 (Earth Engine, Python API) **or**
section 2 (load CSVs exported from the Code Editor script in `notebook 07`).
Both produce the same files:

| file | content |
|---|---|
| `data/climate_monthly.csv` | Oct-Mar 2000-2024, area-averaged over the mountain KRI polygon |
| `data/climate_seasonal.csv` | one row per snow season, with derived predictors |

**Derived predictors built at the end (section 3) - these are the ones that actually carry the
attribution signal:**

- `snowfall_fraction` - snowfall / total precipitation. Separates *"less precipitation"*
  from *"the same precipitation falling as rain"*, which is the single most important
  distinction for a warming semi-arid mountain range.
- `isotherm0_m` - the 0 C isotherm elevation, from a regression of ERA5-Land 2 m temperature
  on SRTM elevation. Directly comparable to the paper's **mean snow elevation (MSE)**, so the
  correlation between the two is a clean physical attribution statement.
- `pdd` - accumulated positive degree-days, the melt-energy term.
- `t2m_djfm`, `precip_ondjfm` - the conventional thermodynamic pair.

In [6]:
import os
import numpy as np
import pandas as pd

ROOT   = os.path.abspath(os.path.join(os.getcwd(), ".."))
OUTDIR = os.path.join(ROOT, "data", "derived")
SHP    = os.path.join(ROOT, "data", "raw", "mountain_Region_KRI.shp")
os.makedirs(OUTDIR, exist_ok=True)
print("shapefile exists:", os.path.exists(SHP))

shapefile exists: True


## Section 1 - Earth Engine Python API (recommended)

```
pip install earthengine-api geopandas
earthengine authenticate
```

Set `EE_PROJECT` to your Cloud project id. If anything here fails, skip to section 2.

In [7]:
EE_PROJECT = "your-earthengine-project-id"      # <-- EDIT
USE_EE     = True                      # set False to use the Code Editor CSVs instead

if USE_EE:
    try:
        import ee, geopandas as gpd, json as _json
        ee.Initialize(project=EE_PROJECT)

        gdf    = gpd.read_file(SHP).to_crs(4326)
        geom   = gdf.union_all() if hasattr(gdf, "union_all") else gdf.unary_union
        region = ee.Geometry(_json.loads(gpd.GeoSeries([geom], crs=4326).to_json())
                             ["features"][0]["geometry"])
        print("region area (km2):", ee.Number(region.area(1000)).divide(1e6).getInfo())
    except Exception as e:
        print("Earth Engine unavailable ->", type(e).__name__, e)
        USE_EE = False

region area (km2): 26165.556828375455


In [8]:
BANDS = ["temperature_2m", "temperature_2m_min", "temperature_2m_max",
         "dewpoint_temperature_2m", "total_precipitation_sum", "snowfall_sum",
         "snow_depth_water_equivalent", "snow_cover"]
SCALE = 1000

if USE_EE:
    era5 = (ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
              .filterDate("2000-10-01", "2024-04-01")
              .filter(ee.Filter.inList("month", [10, 11, 12, 1, 2, 3])))
    dem  = ee.Image("USGS/SRTMGL1_003").rename("elev")

    def month_stats(img):
        stats = img.select(BANDS).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=region,
            scale=SCALE, maxPixels=int(1e13), bestEffort=True)
        # 0 C isotherm elevation from T(z) = a + b*z  ->  z0 = (273.15 - a)/b
        fit = dem.addBands(img.select("temperature_2m")).reduceRegion(
            reducer=ee.Reducer.linearFit(), geometry=region,
            scale=SCALE, maxPixels=int(1e13), bestEffort=True)
        slope, offset = ee.Number(fit.get("scale")), ee.Number(fit.get("offset"))
        return ee.Feature(None, stats).set({
            "date":  img.date().format("YYYY-MM"),
            "year":  img.date().get("year"),
            "month": img.date().get("month"),
            "lapse_K_per_m":    slope,
            "isotherm0_m":      ee.Number(273.15).subtract(offset).divide(slope),
        })

    fc  = ee.FeatureCollection(era5.map(month_stats))
    raw = fc.getInfo()["features"]
    clim = pd.DataFrame([f["properties"] for f in raw])
    clim = clim.sort_values(["year", "month"]).reset_index(drop=True)
    print(clim.shape)
    clim.head(3)

(144, 13)


## Section 2 - fallback: CSVs exported from the Code Editor

Run `notebook 07`, download `ERA5Land_KRI_monthly.csv` and
`ERA5Land_KRI_isotherm.csv` from Drive into `data/derived/`, then run this cell.

*(A third option: download ERA5-Land monthly means from the Copernicus CDS and area-average
them yourself; keep the same column names and everything downstream works unchanged.)*

In [9]:
if not USE_EE:
    m_csv = os.path.join(OUTDIR, "ERA5Land_KRI_monthly.csv")
    i_csv = os.path.join(OUTDIR, "ERA5Land_KRI_isotherm.csv")
    clim  = pd.read_csv(m_csv)
    iso   = pd.read_csv(i_csv)[["year", "month", "lapse_K_per_m", "isotherm0_m"]]
    clim  = clim.merge(iso, on=["year", "month"], how="left")
    clim  = clim.sort_values(["year", "month"]).reset_index(drop=True)
    print(clim.shape)
    clim.head(3)

## Section 3 - unit conversion and derived predictors

ERA5-Land units: temperature in K, precipitation and snowfall in m of water equivalent
per month, SWE in m.

In [10]:
d = clim.copy()

# ---- units
d["t2m"]      = d["temperature_2m"] - 273.15                 # deg C
d["tmin"]     = d["temperature_2m_min"] - 273.15
d["tmax"]     = d["temperature_2m_max"] - 273.15
d["td2m"]     = d["dewpoint_temperature_2m"] - 273.15
d["precip"]   = d["total_precipitation_sum"] * 1000.0        # mm
d["snowfall"] = d["snowfall_sum"] * 1000.0                   # mm w.e.
d["swe"]      = d["snow_depth_water_equivalent"] * 1000.0    # mm w.e.

# ---- derived
d["snowfall_fraction"] = np.where(d["precip"] > 1.0, d["snowfall"] / d["precip"], np.nan)
days = d["month"].map({10: 31, 11: 30, 12: 31, 1: 31, 2: 28.25, 3: 31})
d["pdd"] = np.where(d["t2m"] > 0, d["t2m"] * days, 0.0)      # positive degree-days

# season = Oct(Y)..Mar(Y+1), labelled by ending year (as in notebook 01)
d["season"] = np.where(d["month"] >= 10, d["year"] + 1, d["year"])
d = d[["date", "year", "month", "season", "t2m", "tmin", "tmax", "td2m",
       "precip", "snowfall", "swe", "snow_cover", "snowfall_fraction",
       "pdd", "lapse_K_per_m", "isotherm0_m"]]
d.to_csv(os.path.join(OUTDIR, "climate_monthly.csv"), index=False)
print("saved climate_monthly.csv", d.shape)
d.head(3).round(2)

saved climate_monthly.csv (144, 16)


,date,year,month,season,t2m,tmin,tmax,td2m,precip,snowfall,swe,snow_cover,snowfall_fraction,pdd,lapse_K_per_m,isotherm0_m
0,2000-10,2000,10,2001,15.71,3.41,27.36,2.08,27.11,0.03,0.00,0.01,0.00,487.03,-0.0,4633.03
1,2000-11,2000,11,2001,8.80,-2.49,19.21,-1.04,57.14,3.04,0.33,2.08,0.05,264.06,-0.0,3115.57
2,2000-12,2000,12,2001,3.61,-7.49,12.53,-0.96,138.91,41.85,10.86,24.25,0.30,111.76,-0.0,1938.03


In [11]:
djfm = [12, 1, 2, 3]

agg_mean = ["t2m", "tmin", "tmax", "td2m", "isotherm0_m", "lapse_K_per_m", "snow_cover"]
agg_sum  = ["precip", "snowfall", "pdd"]

s = d.groupby("season").agg({**{c: "mean" for c in agg_mean},
                             **{c: "sum"  for c in agg_sum}})
s.columns = [f"{c}_ondjfm" for c in s.columns]

w = d[d["month"].isin(djfm)].groupby("season").agg({**{c: "mean" for c in agg_mean},
                                                    **{c: "sum"  for c in agg_sum}})
w.columns = [f"{c}_djfm" for c in w.columns]

clim_seasonal = s.join(w).reset_index()
clim_seasonal["snowfall_fraction_ondjfm"] = (clim_seasonal["snowfall_ondjfm"] /
                                             clim_seasonal["precip_ondjfm"])
clim_seasonal["snowfall_fraction_djfm"]   = (clim_seasonal["snowfall_djfm"] /
                                             clim_seasonal["precip_djfm"])
clim_seasonal["swe_max"] = d.groupby("season")["swe"].max().values

clim_seasonal.to_csv(os.path.join(OUTDIR, "climate_seasonal.csv"), index=False)
print("saved climate_seasonal.csv", clim_seasonal.shape)
clim_seasonal.round(2).head()

saved climate_seasonal.csv (24, 24)


,season,t2m_ondjfm,tmin_ondjfm,tmax_ondjfm,td2m_ondjfm,isotherm0_m_ondjfm,lapse_K_per_m_ondjfm,snow_cover_ondjfm,precip_ondjfm,snowfall_ondjfm,...,td2m_djfm,isotherm0_m_djfm,lapse_K_per_m_djfm,snow_cover_djfm,precip_djfm,snowfall_djfm,pdd_djfm,snowfall_fraction_ondjfm,snowfall_fraction_djfm,swe_max
0,2001,7.14,-4.85,18.09,-0.23,2743.35,-0.0,19.91,527.81,141.93,...,-0.60,2177.87,-0.0,29.35,443.56,138.85,559.84,0.27,0.31,33.69
1,2002,6.86,-6.16,18.27,-0.63,2665.26,-0.0,23.36,722.45,153.39,...,-1.30,1999.07,-0.0,31.16,647.02,133.65,492.04,0.21,0.21,46.76
2,2003,6.23,-5.85,17.34,-0.41,2602.90,-0.0,33.24,862.60,307.56,...,-1.77,1610.45,-0.0,49.72,789.29,305.64,278.80,0.36,0.39,132.49
3,2004,7.19,-5.86,18.74,0.47,2801.45,-0.0,25.43,820.08,199.71,...,-0.61,2017.97,-0.0,37.77,622.39,197.21,503.77,0.24,0.32,78.37
4,2005,6.12,-6.57,18.10,-0.83,2513.17,-0.0,34.38,722.22,236.81,...,-2.72,1669.50,-0.0,47.89,531.15,210.36,313.15,0.33,0.40,114.87


In [12]:
# trends in the predictors themselves - these numbers go straight into the Results text
for col in ["t2m_djfm", "precip_ondjfm", "snowfall_ondjfm",
            "snowfall_fraction_ondjfm", "isotherm0_m_djfm", "pdd_ondjfm"]:
    if col not in clim_seasonal:
        continue
    x = clim_seasonal[["season", col]].dropna()
    b, a = np.polyfit(x["season"], x[col], 1)
    print(f"{col:26s} trend = {b:+10.4f} per year   mean = {x[col].mean():10.2f}")

t2m_djfm                   trend =    +0.0631 per year   mean =       4.01
precip_ondjfm              trend =    +2.1426 per year   mean =     706.96
snowfall_ondjfm            trend =    -3.3378 per year   mean =     160.37
snowfall_fraction_ondjfm   trend =    -0.0047 per year   mean =       0.23
isotherm0_m_djfm           trend =   +15.8462 per year   mean =    2022.76
pdd_ondjfm                 trend =   +10.7707 per year   mean =    1312.12


---
**Reporting note.** ERA5-Land is a reanalysis, not an observation. State this explicitly in the
Methods and, if you can obtain even two or three station records from the Kurdistan
meteorological directorate, add a short validation of ERA5-Land 2 m temperature and
precipitation against them. That single paragraph also answers the reviewer's separate
*"no ground-truth validation"* comment.

**Next:** `04_attribution_analysis.ipynb`.